**Figures 7 & 8: PrISM Performance/RFMs vs. PRAC and MINT**

In [ ]:
# ============================================================================
# Figures 7 & 8: PrISM Performance vs. PRAC and MINT
# ============================================================================
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import FancyArrowPatch
from matplotlib.transforms import blended_transform_factory


# ---- Config (edit these freely in the notebook) ----
# Adjust if your notebook lives somewhere other than perf_analysis/plot_scripts/
RESULTS_DIR = Path("../results/collated")
OUTPUT_DIR  = Path("../plots")
TREF        = 2
# ----------------------------------------------------

# X-axis ordering: high-RBMPKI workloads, empty separator, then per-suite GMEANs.
WORKLOAD_ORDER = [
    "429.mcf", "470.lbm", "434.zeusmp", "519.lbm", "549.fotonik3d",
    "459.GemsFDTD", "450.soplex", "462.libquantum", "433.milc",
    "437.leslie3d", "510.parest", "520.omnetpp", "483.xalancbmk",
    "wc_8443", "wc_map0", "482.sphinx3", "tpch2",
    "",  # separator
    "SPEC2K6 (23)", "SPEC2K17 (18)", "TPC (4)",
    "Hadoop (3)", "MediaBench (3)", "YCSB (6)", "All (57)",
]

GEOMEAN_LABELS = {
    "SPEC2K6 (23)", "SPEC2K17 (18)", "TPC (4)",
    "Hadoop (3)", "MediaBench (3)", "YCSB (6)", "All (57)",
}

# (mitigation_column, TRH-D, plot_label)
# QPRAC's overhead is TRH-D-independent in the evaluated regime, so its single
# bar represents all three TRH-D values.
SERIES_SPECS = [
    ("QPRAC", 250,  r"PRAC:$T_{RH-D}=250/500/1000$"),
    ("MINT",  250,  r"MINT+RFM11:$T_{RH-D}=250$"),
    ("MINT",  500,  r"MINT+RFM24:$T_{RH-D}=500$"),
    ("MINT",  1000, r"MINT+RFM48:$T_{RH-D}=1000$"),
    ("PrISM", 250,  r"PrISM:$T_{RH-D}=250$"),
    ("PrISM", 500,  r"PrISM:$T_{RH-D}=500$"),
    ("PrISM", 1000, r"PrISM:$T_{RH-D}=1000$"),
]


# ---- Data prep ----
def build_plot_df(csv_path: Path, value_col_name: str, tref: int):
    """Read a unified CSV and stack rows into the long format seaborn expects.

    The CSV is the condensed (workload, TRH-D, TREF_Freq) format from
    collate.py, with MINT/QPRAC/PrISM as side-by-side columns. PrISM is at
    the default PMQ size (16) already baked into the CSV.
    """
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Could not find {csv_path}. Run collate.py first.")
    df = pd.read_csv(csv_path)

    required = {"workload", "TRH-D", "TREF_Freq"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    rows = []
    for col, trh_d, label in SERIES_SPECS:
        if col not in df.columns:
            print(f"Warning: column '{col}' not present in {csv_path.name}; "
                  f"skipping {label}")
            continue

        mask = (df["TRH-D"] == trh_d) & (df["TREF_Freq"] == tref)
        sub = df.loc[mask, ["workload", col]].copy()
        if sub.empty:
            print(f"Warning: no rows for {label} "
                  f"(col={col}, TRH-D={trh_d}, TREF={tref})")
            continue

        sub = sub.rename(columns={col: value_col_name})
        sub["Methods"] = label
        rows.append(sub)

    df_long = pd.concat(rows, ignore_index=True)
    df_long = df_long[df_long["workload"].isin(WORKLOAD_ORDER)].copy()

    method_order = [label for _, _, label in SERIES_SPECS]
    df_long["Methods"] = pd.Categorical(df_long["Methods"],
                                        categories=method_order, ordered=True)
    return df_long, method_order


# ---- Group-box decoration ----
def draw_group_box(ax, transform, start_idx, end_idx, sep_x, label, side,
                   arrow_y, box_y, box_height, fontsize=11,
                   outer_extend=1.4, inner_gap=0.0, box_pad=0.35):
    """Arrow + labeled box under the x-axis spanning [start_idx, end_idx]."""
    raw_start = start_idx + 0.8
    raw_end   = end_idx - 0.8

    if side == "left":
        arrow_start, arrow_end = raw_start - outer_extend, sep_x - inner_gap
    elif side == "right":
        arrow_start, arrow_end = sep_x + inner_gap, raw_end + outer_extend
    else:
        raise ValueError("side must be 'left' or 'right'")

    box_start, box_end = raw_start, raw_end
    box_center = (box_start + box_end) / 2

    ax.add_patch(FancyArrowPatch(
        posA=(arrow_start, arrow_y), posB=(arrow_end, arrow_y),
        arrowstyle="<->", mutation_scale=12, linewidth=1.2,
        transform=transform, color="black", clip_on=False, zorder=1,
    ))
    ax.add_patch(patches.FancyBboxPatch(
        (box_start + box_pad, box_y),
        (box_end - box_start) - 2 * box_pad, box_height,
        boxstyle="round,pad=0.02", linewidth=1.2,
        edgecolor="black", facecolor="white",
        transform=transform, clip_on=False, zorder=2,
    ))
    ax.text(box_center, box_y + box_height / 2, label,
            ha="center", va="center", fontsize=fontsize,
            transform=transform, zorder=3)


# ---- Figure rendering ----
def render_figure(df_long, method_order, value_col, ylabel,
                  ylim, yticks, mean_label, mean_label_y,
                  out_path, draw_ellipse=False):
    """Render one bar-chart figure with the standard group-box decoration."""
    sns.set_palette("tab10")
    sns.set_style("whitegrid")
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"]  = 42

    fig, ax = plt.subplots(figsize=(12, 2))
    plt.rc("font", size=10)

    sns.barplot(
        x="workload", y=value_col, hue="Methods",
        data=df_long, order=WORKLOAD_ORDER, hue_order=method_order,
        edgecolor="black", ax=ax,
    )

    ax.set_xticks(np.arange(len(WORKLOAD_ORDER)))
    ax.set_xticklabels(WORKLOAD_ORDER, ha="right", rotation=45, fontsize=11)
    for tick in ax.get_xticklabels():
        if tick.get_text() in GEOMEAN_LABELS:
            tick.set_fontweight("bold")

    sep_idx = WORKLOAD_ORDER.index("")
    gmean_center = (sep_idx + 1 + len(WORKLOAD_ORDER) - 1) / 2

    ax.axvline(sep_idx, 0, 1, color="red", linestyle="--", linewidth=2)
    ax.text(gmean_center, mean_label_y, mean_label, fontweight="bold", ha="center")

    ax.tick_params(axis="x", which="major", labelsize=10.5)
    ax.tick_params(axis="y", which="major", labelsize=11.5)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=11)

    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.42),
              ncol=4, fancybox=True, shadow=False, fontsize=10)

    ax.set_ylim(*ylim)
    ax.set_xlim(-0.5, len(WORKLOAD_ORDER) - 0.5)
    if yticks is not None:
        ax.set_yticks(yticks)

    # Optional Fig. 7 ellipse around the "0.6" tick + horizontal line at 1.0
    if draw_ellipse:
        ax.add_patch(patches.Ellipse(
            (-1.1, ylim[0]), width=0.8, height=0.07,
            edgecolor="red", fill=False, clip_on=False,
            facecolor="none", linewidth=1.5,
        ))
        ax.axhline(y=1.0, color="r", linestyle="-", linewidth=2)

    # Group-box decoration under the x-axis
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    arrow_y, box_y, box_height = -0.82, -0.86, 0.08

    draw_group_box(
        ax, transform,
        start_idx=0, end_idx=sep_idx - 1, sep_x=sep_idx,
        label="Workloads with \u2265 10 Row-Buffer Misses per Kilo-Instruction",
        side="left", arrow_y=arrow_y, box_y=box_y, box_height=box_height,
        outer_extend=1.4, inner_gap=0.0, box_pad=0.25,
    )
    draw_group_box(
        ax, transform,
        start_idx=sep_idx + 1, end_idx=len(WORKLOAD_ORDER) - 1, sep_x=sep_idx,
        label="All Workloads",
        side="right", arrow_y=arrow_y, box_y=box_y, box_height=box_height,
        outer_extend=1.4, inner_gap=0.0, box_pad=0.35,
    )

    plt.grid(True, linestyle=":")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=600, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()  # display inline in the notebook


# ---- Render Figure 7: normalized performance ----
perf_df, method_order = build_plot_df(
    RESULTS_DIR / "perf_normalized.csv",
    value_col_name="WS", tref=TREF,
)
render_figure(
    perf_df, method_order,
    value_col="WS",
    ylabel="Normalized Performance",
    ylim=(0.6, 1.05),
    yticks=[0.6, 0.7, 0.8, 0.9, 1.0],
    mean_label="GMEAN", mean_label_y=1.02,
    out_path=OUTPUT_DIR / "fig7_perf.pdf",
    draw_ellipse=True,
)

# ---- Render Figure 8: RFMs per tREFI ----
rfm_df, _ = build_plot_df(
    RESULTS_DIR / "rfm_per_trefi.csv",
    value_col_name="RFM", tref=TREF,
)
ymax = rfm_df["RFM"].max()
mean_label_y = ymax * 0.85 if pd.notna(ymax) and ymax > 0 else 7
render_figure(
    rfm_df, method_order,
    value_col="RFM",
    ylabel="RFMs per tREFI per Channel",
    ylim=(0, max(ymax * 1.05, 1) if pd.notna(ymax) else 12),
    yticks=None,
    mean_label="AMEAN", mean_label_y=mean_label_y,
    out_path=OUTPUT_DIR / "fig8_rfm.pdf",
    draw_ellipse=False,
)

**Figure 9: TRR rate sensitivity at TRH-D = 500**

In [ ]:
# ============================================================================
# Figure 9: TRR rate sensitivity at TRH-D = 500
# ============================================================================
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# ---- Config (edit these freely in the notebook) ----
# Adjust if your notebook lives somewhere other than perf_analysis/plot_scripts/
RESULTS_DIR = Path("../results/collated")
OUTPUT_DIR  = Path("../plots")
WORKLOAD    = "All (57)"
TRH_D       = 500
# ----------------------------------------------------

# X-axis: TRR rate (one TRR per X tREFIs). 0 = no TRR.
X_TICKS = [1, 2, 4, 8, 0]
X_TICK_LABELS = {1: "1", 2: "2", 4: "4", 8: "8", 0: "No"}

# (mitigation_column, plot_label)
# At TRH-D=500, MINT uses one RFM per 24 activations.
SERIES_SPECS = [
    ("QPRAC", "PRAC"),
    ("MINT",  "MINT+RFM24"),
    ("PrISM", "PrISM"),
]

PALETTE = sns.color_palette("tab10")
METHOD_COLORS = {
    "PRAC":        PALETTE[0],
    "MINT+RFM24":  PALETTE[2],
    "PrISM":       PALETTE[5],
}


# ---- Data prep ----
def build_plot_df(csv_path: Path, workload: str, trh_d: int) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Could not find {csv_path}. Run collate.py first.")
    df = pd.read_csv(csv_path)

    required = {"workload", "TRH-D", "TREF_Freq"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {csv_path}: {missing}")

    rows = []
    for col, label in SERIES_SPECS:
        if col not in df.columns:
            print(f"Warning: column '{col}' not in {csv_path.name}; skipping {label}")
            continue

        mask = (
            (df["workload"] == workload)
            & (df["TRH-D"] == trh_d)
            & (df["TREF_Freq"].isin(X_TICKS))
        )
        sub = df.loc[mask, ["TREF_Freq", col]].copy()
        if sub.empty:
            print(f"Warning: no rows for {label} "
                  f"(col={col}, workload={workload}, TRH-D={trh_d})")
            continue

        sub = sub.rename(columns={col: "WS"})
        sub["Mitigations"] = label
        rows.append(sub.dropna(subset=["WS"]))

    if not rows:
        raise ValueError("No data matched the selected filters.")

    df_long = pd.concat(rows, ignore_index=True)
    df_long = (df_long
               .groupby(["Mitigations", "TREF_Freq"], as_index=False)["WS"]
               .mean())
    return df_long


# ---- Render ----
df_long = build_plot_df(RESULTS_DIR / "perf_normalized.csv",
                        workload=WORKLOAD, trh_d=TRH_D)

methods_in_order = [
    label for _, label in SERIES_SPECS
    if label in set(df_long["Mitigations"])
]

sns.set_palette("tab10")
sns.set_style("whitegrid")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42

fig, ax = plt.subplots(figsize=(8, 2))
plt.rc("font", size=12)

bar_width = 0.12
num_bars = len(methods_in_order)
x_tick_positions = np.arange(len(X_TICKS), dtype=float)

bar_positions = {
    tick: [base - (bar_width * num_bars) / 2 + j * bar_width
           for j in range(num_bars)]
    for tick, base in zip(X_TICKS, x_tick_positions)
}

for tick in X_TICKS:
    sub = df_long[df_long["TREF_Freq"] == tick]
    for i, method in enumerate(methods_in_order):
        row = sub[sub["Mitigations"] == method]
        if row.empty:
            continue
        value = row["WS"].iloc[0]
        x_pos = bar_positions[tick][i] + bar_width / 2
        ax.bar(
            x_pos, value,
            width=bar_width,
            color=METHOD_COLORS.get(method, "gray"),
            edgecolor="black",
            label=method if tick == X_TICKS[0] else "",
        )

for i in range(len(X_TICKS) - 1):
    ax.axvline(x=i + 0.5, color="grey", linestyle="-", alpha=0.5)

handles, labels = ax.get_legend_handles_labels()
seen = dict(zip(labels, handles))
ax.legend(
    seen.values(), seen.keys(),
    loc="upper center", bbox_to_anchor=(0.5, 1.25),
    ncol=4, fancybox=True, shadow=False, fontsize=11,
)

ax.set_xticks(x_tick_positions)
ax.set_xticklabels([X_TICK_LABELS[t] for t in X_TICKS])
ax.set_xlabel(r"Target Row Refresh (TRR) Rate (One TRR per X $\mathrm{tREFI}$)",
              fontsize=12)
ax.set_ylabel("Normalized Performance", fontsize=12)
ax.tick_params(axis="both", which="major", labelsize=12)

ax.axhline(y=1.0, color="r", linestyle="-", linewidth=2)
ax.set_ylim(0.8, 1.008)
ax.set_yticks([0.8, 0.85, 0.9, 0.95, 1.0])
ax.set_xlim(-0.5, len(X_TICKS) - 0.5)

ax.add_patch(patches.Ellipse(
    (-0.73, 0.8), width=0.34, height=0.04,
    edgecolor="red", fill=False, clip_on=False,
    facecolor="none", linewidth=1.5,
))

plt.grid(True, linestyle=":")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / "fig9_trr_sensitivity.pdf"
fig.savefig(out_path, dpi=600, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

**Table VI: PMQ size sensitivity at TRH-D = 500**

In [ ]:
# ============================================================================
# Table VI: PMQ size sensitivity at TRH-D = 500
# ============================================================================
from pathlib import Path

import pandas as pd


# ---- Config (edit these freely in the notebook) ----
# Adjust if your notebook lives somewhere other than perf_analysis/plot_scripts/
RESULTS_DIR    = Path("../results/collated")
OUTPUT_DIR     = Path("../tables")
PMQ_SIZES      = [4, 8, 16, 32]
WORKLOAD       = "All (57)"
HIGHLIGHT_PMQ  = 16   # PMQ size to mark with * (paper default)

# Worst-case slack activations during chained ABO, per Section IV-B of the
# PrISM paper. Add new entries here if you sweep additional PMQ sizes.
ABO_ACT_TABLE = {
    4:  7,
    8:  10,
    16: 12,
    32: 14,
}
# ----------------------------------------------------


def abo_act_for_pmq(q: int) -> int:
    if q not in ABO_ACT_TABLE:
        raise ValueError(
            f"ABO_ACT(Q={q}) not in the published table. "
            f"Known PMQ sizes: {sorted(ABO_ACT_TABLE.keys())}. "
            f"Add the value to ABO_ACT_TABLE if you've derived it for a new PMQ size."
        )
    return ABO_ACT_TABLE[q]


def collect_overheads(csv_path: Path) -> dict:
    """Return {pmq_size: overhead_percent} from pmq_sweep.csv."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find {csv_path}. Run collate.py first.")
    df = pd.read_csv(csv_path)

    if "workload" not in df.columns:
        raise ValueError(f"Missing 'workload' column in {csv_path}")

    pmq_cols = [f"PMQ{p}" for p in PMQ_SIZES]
    missing = set(pmq_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing PMQ columns in {csv_path}: {missing}")

    row = df.loc[df["workload"] == WORKLOAD]
    if row.empty:
        raise ValueError(
            f"Workload '{WORKLOAD}' not found in {csv_path}. "
            f"Available workloads: {df['workload'].unique().tolist()[:10]}...")

    overheads = {}
    for pmq, col in zip(PMQ_SIZES, pmq_cols):
        value = row[col].iloc[0]
        if pd.isna(value):
            print(f"Warning: no PrISM value for workload={WORKLOAD}, PMQ={pmq}")
            overheads[pmq] = None
        else:
            overheads[pmq] = (1.0 - float(value)) * 100.0
    return overheads


def format_overhead(value) -> str:
    return f"{value:.1f}%" if value is not None else "--"


def render_table(overheads: dict) -> str:
    header = ("PMQ Size", "ABO_ACT(Q)", "Perf. Overhead")
    rows = []
    for pmq in PMQ_SIZES:
        abo = abo_act_for_pmq(pmq)
        oh = format_overhead(overheads.get(pmq))
        marker = " *" if pmq == HIGHLIGHT_PMQ else "  "
        rows.append((f"{pmq}{marker}", str(abo), oh))

    col_widths = [max(len(header[i]), max(len(r[i]) for r in rows)) for i in range(3)]
    sep = "+-" + "-+-".join("-" * w for w in col_widths) + "-+"
    fmt = "| " + " | ".join(f"{{:<{w}}}" for w in col_widths) + " |"

    out = [sep, fmt.format(*header), sep]
    for r in rows:
        out.append(fmt.format(*r))
    out.append(sep)
    out.append("")
    out.append("(* = paper default)")
    return "\n".join(out) + "\n"


# ---- Run ----
overheads = collect_overheads(RESULTS_DIR / "pmq_sweep.csv")
table = render_table(overheads)

print(table)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / "table6_pmq_sensitivity.txt"
out_path.write_text(table)
print(f"Saved: {out_path}")